In [1]:
from gensim.models import Word2Vec
import numpy as np
from collections import Counter

In [2]:
# paths to files
clear_file_path = "clear_lines.txt"
ciphered_file_path = "ciphered_lines.txt"
solutions_file_path = "ciphered_lines_ground_truth.txt"

In [3]:
corpus_clear = [line.strip().lower() for line in open(clear_file_path)]
corpus_ciphered = [line.strip().lower() for line in open(ciphered_file_path)]
corpus_ground_truth = [line.rstrip('\n').lower() for line in open(solutions_file_path)]

In [16]:
# this is the text to decipher
corpus_ground_truth[:10]

['a tajną . u rzymian niewolnika , który ukradł , ',
 'środka : tam wszystko jasno oświetlone . * i widzę ',
 'się do góry , utracił własność * dążącą do centru ',
 'chciałabym widzieć tego malca tylko na chwilę . słyszałem , ',
 'mości notę , prosząc o uwolnienie z tej usługi , ',
 '? — zagadnął mnie mój przewodnik . odpowiedziałem machinalnie , ',
 ', choroba poplątała jej szyki ; już w polsce , ',
 'mąż córkę jednego z sąsiadów . ja byłem drużbą . ',
 'cudzoziemiec nie mógłbym otrzymać w tym smutnym skądinąd kierunku jakichkolwiek ',
 'passavent sądzi przeciwnie , że jest w gruncie bardzo łagodny ']

In [15]:
# ciphered ground truth
corpus_ciphered[:10]

['🥥🖕👼🥥🔀🛠👡🐌💸🐌📩🖕🧇🤿🧰🦆🥨🥥🦨🐌🚕🔻🥅🐘🚒🗣🛠🔻🚄🥥🤳🥺🐌🚄👼👲🐤🧰🤳📩🚄🔒🥥🧵🐈🖕🥺🖕',
 '🥤🔒🚒🚔🚄🦫🐌🦒🐌👼🦫🦆🖕🐘🦓🤿💣🦓🚷👱🧔🖕💐🥥🚭🚕🚒🐌🙉🥤🐘🥨🗿👼🗣🚒🦨🔶🤳💑🖕📈🐌🔻🖕🖊🔻🚔🐅👽🐌',
 '🦓🔻👽🐌🧵🚒🤳🚥🐛🐤💁🖕🖈🤳🔖👼🧇🦫💯🕰👂🤳🐘👂🥥🚭🛠🚒🚚🛰🤳🧚🐌🧵👠🦔🕶🦈👡🐌🚓🙉🐌🦈🗿🦨🚷🐤🧍🐌',
 '🚼😁💯🥨🛟🐈🥥🛃💁🦆🖕🖊🔻🚔🐅🔻🥅🛰🖕👼🗿📉🧔🐌🗺🦫🕳💯🦫🖕🚷💁🗝🚄🚒🤳🛠🥥🐌🦈😁🐘🥨🗝👽🤳👛🤳🦓👂💁🦓🤿🦫👂🔶📸🖕🥺🤳',
 '🗺🙉🔔🚼🕰🖕🛠🚒📺👽🐌🖈🐌🤱🔒🚒🦓🐅👡💯🐌🚒🖕🔖🐘🚒🕳🛠🔻🔶🛠🔻🔶🤳🤿🤳📺🥅🦚🐌🧍🦓🐈🧍🚥🔻🐌🖈🐌',
 '💮🐌🦱🐌🐅🦫🛣🥥🧵🛠🕶👂🐌📸🚕🔻🥅🖕📸👲🔀🤳🥳🧇🧠🔶🖊🙉🚔🚕🔻🚄🤳💸🐌🚒🚓🥳🧔🥊🔻🥅🧵🤿🔻🛟👂🔶🗺🖕🦆🦫🚼😁🕰🛠🥥🗣🛠🕰🗿🤳🥺🐌',
 '🥺🐌🦈🚳🚒🧇🚒📭🥥🖕🥳🧔👦🗣👠📺🛟🐈🛟🖕🦚🥅💐🤳🚭🤿🧰🚄🕰🤳🧉🐌💐🧍🐿🐌🖊🐌🥳🚒🗝🦓💯🥅🐌🖈🖕',
 '🗺👡🦔🖕🦈🐛🔒👱👽🐌💐🥅🚔🦨🥅🛣🚒🤳🧠🤳🦓👡🦓🕰🥥🚔🐛🥊🖕💸🖕🦚🥥🐌🛃💣👂🥅🗺🐌🚔🔒🧍🥈📟👡🐌💸🐌',
 '💯🧍🚔🐅🚒🐅🕰🔶📸🥨🥅🚼🖕🛠🥨🗿🐌🦆📎🚥🐈🛃💣📸🤳🚒🚷🔒🤿💁🦆🛟🖎🐌🐘🐌👼🧰🗺🐌🚭🦆📩👼🛠🧰🦆🖕🦓💶👡🚔🔻🛠👠🚓🐌🚄🕰🥅🐤📩🦨🚄📩🐌🦚🛟🚄🔻🦈😁👱🚒🗣🥊🥨🥅🚄🐌',
 '🥳🥥🦓🦓🛟🔹🗿🛠📺🤳🦓👡🚓🤿🕰🖕🤱🔒🤿🔶🦈🔻🐘🛠🕰🥅🐌🖈🖕🦔🔶🤳💐🔶🦓📺🐌🐘🐌📉🧇🧍🛠💯🥨🥅🐌🛃🛟🧇🧵🐅🚒🐌👂🦫📉🙉🚓🦨💣🐌']

In [6]:
# this is to find frequency of each character
corpus_clear[:5]

['mieszkańca prowincji ( cel jezuitów w tworzeniu nowej moralności .',
 'na poddaszu i przypomniała sobie znowu kaja . — och',
 'sensacyjnych momentach : „ słuchajcie , słuchajcie ” . i',
 'parisien ” pisze , że francuzi nie będą tym zachwyceni',
 'co ich gnębi . jeść obiad razem , rzucać przy']

In [7]:
# similiarity function between 2 vectors
def cosine_similarity(vector_1, vector_2):
    dot_product = np.dot(vector_1, vector_2)
    
    norm_1 = np.linalg.norm(vector_1)
    norm_2 = np.linalg.norm(vector_2)

    cos_similarity = dot_product / (norm_1 * norm_2)

    return cos_similarity

In [8]:
# O(n^2 log n)    where n is the amount of different emojis (less than 300)

def divide_into_groups(wv, threshold):

    # all emojis
    emojis = []
    for emoji in wv.index_to_key:
        emojis.append(emoji)
    emojis.reverse()

    groups = {} # groups (same group same letter represented)
    idx = 0 # idxs of groups
    
    while len(emojis) > 0:
        emoji1 = emojis.pop()
        groups[emoji1] = idx

        similiarities = []
        for emoji2 in emojis:
            vector1 = np.array(wv[emoji1])
            vector2 = np.array(wv[emoji2])

            similiarity = cosine_similarity(vector1, vector2)
            similiarities.append((similiarity, emoji2))
        
        similiarities.sort(reverse = True)

        # 3 characters can represent one letter
        for similiarity, emoji2 in similiarities[:2]: 
            if(similiarity >= threshold):
                groups[emoji2] = idx
                emojis.remove(emoji2)
        idx += 1
    return groups, idx

In [9]:
# get all frequencies of each letter from the clear corpus
def get_char_frequency_map(clear_corpus):
    char_counter = Counter()
    
    for line in clear_corpus:
        char_counter.update(line)
    sorted_char_frequency = sorted(char_counter.items(), key=lambda x: x[1], reverse=True)
    
    return sorted_char_frequency

# get all frequencies of each deciphered letter (some emojis)
def get_group_frequencies(char_frequency, groups):
    group_sums = Counter()

    for char, freq in char_frequency:
        group = groups[char]
        group_sums[group] += freq

    return dict(group_sums)

In [10]:
def decipher_corpus(clear_corpus, ciphered_corpus):
    # TODO: Zaimplementuj funkcję odszyfrowującą teksty z `ciphered_corpus` i zwracającą odszyfrowane teksty
    
    unique_letters = set()
    for line in clear_corpus:
        for letter in line:
            unique_letters.add(letter)
    N = len(unique_letters) # number of unique letters
    
    l2v = Word2Vec(ciphered_corpus, vector_size = 24, window = 5, workers = 16, min_count = 0, epochs = 40)

    # binsearch the threshold
    a = -1.0
    b = 1.0
    while b - a > 0.0000001:
        threshold = (a + b) / 2
        groups, num_groups = divide_into_groups(l2v.wv, threshold)
        if num_groups <= N:
            a = threshold
        else:
            b = threshold
    groups, num_groups = divide_into_groups(l2v.wv, a)
    
    # get frequencies
    char_frequency_clear = get_char_frequency_map(corpus_clear)
    char_frequency_ciphered = get_char_frequency_map(ciphered_corpus)
    group_sums = get_group_frequencies(char_frequency_ciphered, groups)

    group_sums = dict(sorted(group_sums.items(), key=lambda item: item[1], reverse=True))

    mapping = dict()
    i = 0
    for idx, value in group_sums.items():
        letter, _ = char_frequency_clear[i]
        print(idx, value, letter)
        i += 1
        mapping[idx] = letter

    # finally get deciphered text
    deciphered = []
    for line in ciphered_corpus:
        deciphered_line = []
        for letter in line:
            idx = groups[letter]
            deciphered_line.append(mapping[idx])
        deciphered.append(deciphered_line)
        
    return deciphered

In [11]:
ans = decipher_corpus(corpus_clear, corpus_ciphered)

0 300000  
1 115762 a
2 114754 i
4 105459 e
3 94498 o
5 77345 z
8 69435 n
7 55756 r
9 55694 s
6 54234 w
11 52620 y
12 52098 c
10 48584 t
16 44473 d
14 43357 m
13 39866 k
15 36686 p
17 31748 ł
18 30920 j
23 29130 u
20 27133 ,
19 26947 l
21 23455 b
22 19542 g
25 19203 ę
24 15553 ą
26 13900 h
30 13557 ż
27 13507 .
28 11438 ó
29 11297 ś
31 7932 ć
32 5148 —
33 2570 f
34 2310 ;
35 2118 ń
36 1558 !
38 1534 ?
37 1510 :
39 1021 ź
40 969 „
41 968 ”
44 839 *
42 782 …
45 572 -
43 549 v
52 262 1
51 259 /
46 255 )
53 224 (
54 194 x
47 186 '
48 183 é
49 152 8
50 146 0
56 126 2
57 121 7
61 120 3
62 115 q
58 105 6
63 97 9
55 82 5
66 78 4
67 63 è
59 55 ê
60 46 «
64 38 »
68 37 [
65 33 ]
70 28 à
69 18 ç
71 17 °
72 16 –
73 16 â
74 12 ô
75 11 ü
76 11 ö
77 11 §
78 8 ë
82 8 î
81 7 α
79 7 š
80 7 ä
85 6 č
84 6 ’
83 6 ε
94 6 ο
97 6 ù
89 5 á
88 5 ι
87 5 ї
90 5 +
86 5 %
99 5 í
92 4 ν
93 4 ς
91 4 υ
95 3 δ
96 3 ρ
135 3 ‘
137 3 ½
123 3 ω
110 3 λ
100 2 β
101 2 =
102 2 κ
98 2 σ
134 2 œ
107 2 û
127 1 ï
125 1 †
111 1 τ
1

In [35]:
for i in range(10):
    print(''.join(ans[i]))

a tajną . u szymian niewo,nika l któsy uksadł l 
śsodka ? tam wrzyrtko jarno oświet,one . * i widzę 
rię do gósy l utsacił włarność * dążącą do centsu 
chciałabym widzieć tego ma,ca ty,ko na chwi,ę . rłyrzałem l 
mości notę l psorząc o uwo,nienie z tej urługi l 
: — zagadnął mnie mój pszewodnik . odpowiedziałem machina,nie l 
l chosoba pop,ątała jej rzyki ; już w po,rce l 
mąż cóskę jednego z rąriadów . ja byłem dsużbą . 
cudzoziemiec nie mógłbym otszymać w tym rmutnym rkądinąd kiesunku jakichko,wiek 
parra-ent rądzi pszeciwnie l że jert w gsuncie basdzo łagodny 


In [36]:
corpus_ground_truth[:10]

['a tajną . u rzymian niewolnika , który ukradł , ',
 'środka : tam wszystko jasno oświetlone . * i widzę ',
 'się do góry , utracił własność * dążącą do centru ',
 'chciałabym widzieć tego malca tylko na chwilę . słyszałem , ',
 'mości notę , prosząc o uwolnienie z tej usługi , ',
 '? — zagadnął mnie mój przewodnik . odpowiedziałem machinalnie , ',
 ', choroba poplątała jej szyki ; już w polsce , ',
 'mąż córkę jednego z sąsiadów . ja byłem drużbą . ',
 'cudzoziemiec nie mógłbym otrzymać w tym smutnym skądinąd kierunku jakichkolwiek ',
 'passavent sądzi przeciwnie , że jest w gruncie bardzo łagodny ']

In [30]:
for line1, line2 in zip(ans, corpus_ground_truth):  
    good = 0
    all = 0
    for letter1, letter2 in zip(line1, line2):
        if letter1 == letter2:
            good += 1
        all += 1
print(f'Accuracy of deciphering: {good / all}')

Accuracy of deciphering: 0.9315068493150684
